In [ ]:
import torch
import torchvision
from torchvision import datasets
import matplotlib.pyplot as plt

In [ ]:
cifar10 = datasets.CIFAR10("./cifar10/", train=True,download=False)
cifar10_val = datasets.CIFAR10("./cifar10_val/", train=False, download=False)

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(3072, 512),
    nn.Tanh(),
    nn.Linear(512,2),
    nn.LogSoftmax(dim=1),
)

loss_fn = nn.NLLLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
toTensor = torchvision.transforms.ToTensor()

tensor_cifar10 = [toTensor(img) for img, _ in cifar10]
cifar2 = [(toTensor(img), label-1) for img, label in cifar10 if label in (1,2)]
cifar2_val = [(toTensor(img), label-1) for img, label in cifar10_val if label in (1,2)]

# imgs = torch.stack([img for img, _ in cifar2])
train_cifar2 = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=False)
# labels = torch.tensor([label for _, label in cifar2])

imgs_val = torch.stack([img for img, _ in cifar2_val])
labels_val = torch.tensor([label for _, label in cifar2_val])

In [ ]:
def training(epoch):
    for n in range(epoch):
        for img, label in train_cifar2:
            batch = img.shape[0]

            x = img.view(batch, -1)
            p = model(x)
            loss = loss_fn(p, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        with torch.no_grad():
            pv= model(imgs_val.view(imgs_val.shape[0], -1))
            loss_val = loss_fn(pv, labels_val)
            
        if epoch % 10 == 0:
            print(f"Training loss = {loss}, Validation loss = {loss_val}")

training(100)
            

In [ ]:
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle=True)
total = 0
count = 0
with torch.no_grad():
    for imgs, labels in val_loader:
        batch = imgs.shape[0]

        p = model(imgs.view(batch, -1))

        _, out = torch.max(p, dim=1)
        
        total += batch
        count += int((out == labels).sum())
    
    print(f"accuracy = {count / total}")




In [ ]:
# c_mean, c_std = torch.zeros(3), torch.zeros(3)
# for index, c in enumerate(img):
#     c_m = torch.mean(c)
#     c_s = torch.std(c)

#     c_mean[index] = c_m
#     c_std[index] = c_s

# c_mean, c_std

# transformer = torchvision.transforms.Normalize(c_mean, c_std)
# t_img = transformer(r_img)

# c_m_mean = r_img.view(3,-1).mean(dim=1)
# c_m_std = r_img.view(3,-1).std(dim=1)

# transformer2 = torchvision.transforms.Normalize(c_m_mean, c_m_std)
# t_m_img = transformer(r_img)

# fig, axes = plt.subplots(1, 3, figsize=(10, 5))
# axes[0].imshow(t_img.permute(1,2,0))
# axes[1].imshow(r_img.permute(1,2,0))
# axes[2].imshow(t_m_img.permute(1,2,0))
# for ax in axes:
#     ax.axis('off')
# plt.tight_layout()
# plt.show()